# EDA des ventes NordRetail avec pandas

**Complement pratique du module** `04-Analyse-Exploratoire-EDA`.

Le gerant de la Ch'ti Boutique t'a tendu une cle USB : *« Voila toutes les ventes... je sens qu'il y a un truc qui cloche. Tu peux regarder ? »*

Ta mission : faire parler ces donnees. On suit la demarche EDA en 6 etapes : **charger &rarr; decouvrir &rarr; nettoyer &rarr; decrire &rarr; visualiser &rarr; conclure**.

> Ce notebook est **executable** : lance chaque cellule (Maj+Entree) dans l'ordre.

## 1. Charger les donnees

Premier geste : les imports (`pd` est la convention universelle) et le chargement du vrai fichier `ventes_magasins.csv` avec `read_csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Chargement du vrai dataset NordRetail (chemin relatif depuis ce notebook)
ventes = pd.read_csv("../../99-Brief/Data-Analyst/data/ventes_magasins.csv")

ventes.head()

## 2. Decouvrir un fichier inconnu

Avant **tout** calcul, on regarde a quoi ressemblent les donnees. Le reflexe des 10 premieres minutes : `shape`, `info`, `dtypes`.

> Dans `info()`, compare le *Non-Null Count* au nombre de lignes (valeurs manquantes) et surveille les `Dtype = object` sur des colonnes qui devraient etre des dates ou des nombres. Ici, la colonne `date` est en `object` (du texte) : on la corrigera a l'etape Nettoyer.

### 3. Qualite des donnees : valeurs manquantes et doublons

Juste apres, on controle la qualite. Un `NaN` (case vide) fausse les calculs : on les compte par colonne avec `isna().sum()`, puis en pourcentage. On verifie aussi les lignes en double avec `duplicated()` (les deux cellules de code ci-dessous : `info()` d'abord, puis le comptage des manquants et doublons).

In [ ]:
print("Forme (lignes, colonnes) :", ventes.shape)
print("\nColonnes :", list(ventes.columns))

ventes.info()

In [ ]:
# Nombre de valeurs manquantes par colonne
print("Valeurs manquantes par colonne :")
print(ventes.isna().sum())

# Taux de manquants en %
print("\nTaux de manquants (%) :")
print((ventes.isna().mean() * 100).round(1))

# Lignes entierement dupliquees
print("\nNombre de lignes en double :", ventes.duplicated().sum())

> **Bonne nouvelle qualite.** Ici le fichier est propre : 0 valeur manquante et 0 doublon. C'est un bon signe, mais le geste reste indispensable : sur un fichier reel recu par mail, c'est souvent la que se cachent les problemes.

### Les 3 strategies face aux NaN (a connaitre pour le jour ou il y en aura)

| Strategie | Code | Quand |
|---|---|---|
| Supprimer les lignes | `df.dropna(subset=["montant"])` | Peu de manquants (<5 %) et valeur essentielle |
| Remplir par une modalite | `df["categorie"].fillna("Inconnu")` | Variable qualitative |
| Remplir par une stat | `df["montant"].fillna(df["montant"].median())` | Variable quantitative (mediane robuste) |

> Pense a **reaffecter** : `df = df.dropna(...)`, sinon rien n'est modifie.

## 4. Nettoyage leger

On corrige le seul vrai probleme detecte : la `date` en texte &rarr; vraie date. On homogeneise la casse de `ville`, on applique la strategie de remplissage sur `categorie` (inoffensif ici, mais bon reflexe) et on retire d'eventuels doublons.

In [ ]:
# Convertir la date texte -> datetime (rend les donnees calculables dans le temps)
ventes["date"] = pd.to_datetime(ventes["date"], format="%Y-%m-%d")

# Homogeneiser la casse des villes
ventes["ville"] = ventes["ville"].str.strip().str.capitalize()

# Etiqueter d'eventuelles categories vides + retirer les doublons (defensif)
ventes["categorie"] = ventes["categorie"].fillna("Inconnu")
ventes = ventes.drop_duplicates()

# Verification : la date est maintenant datetime64
print(ventes.dtypes)
print("\nVilles apres nettoyage :", ventes["ville"].unique())

## 5. Decrire : statistiques descriptives

`describe()` donne d'un coup moyenne (`mean`), ecart-type (`std`), quartiles (`25% / 50% / 75%`) et extremes. C'est ton Chapitre 3 de maths, industrialise sur 12 000 lignes.

> **Devine avant de regarder.** Moyenne et mediane du `montant` seront-elles proches ? Y aura-t-il un max delirant ?

In [ ]:
# Stats descriptives des colonnes numeriques
ventes[["quantite", "prix_unitaire", "remise", "montant", "marge"]].describe().round(2)

> **Lecture.** La `mean` du `montant` est superieure a la mediane (`50%`) : la distribution est **etiree vers la droite** par quelques gros paniers. Le `max` tres au-dessus du 3e quartile signale des valeurs a investiguer (outliers). Pour parler du panier « typique » au gerant, on privilegiera la **mediane**.

## 6. Agreger avec groupby

Le coeur de l'analyse : *decouper &rarr; calculer &rarr; recombiner*. On range les tickets par paquets (par ville) puis on calcule sur chaque paquet. On enchaine ensuite directement sur la **visualisation** de ces agregats.

In [ ]:
# Synthese par ville : CA total, nombre de ventes, panier median
synthese = (ventes.groupby("ville")["montant"]
            .agg(CA="sum", nb_ventes="count", panier_median="median")
            .sort_values("CA", ascending=False))
print(synthese.round(2))

# CA par categorie de produit
print("\nCA par categorie :")
print(ventes.groupby("categorie")["montant"].sum().sort_values(ascending=False).round(2))

## 7. Visualiser (3 graphiques matplotlib)

Un graphique revele en un clin d'oeil ce qu'un tableau cache : **histogramme** (forme d'une distribution), **barres** (comparer des categories), **boxplot** (reperer les outliers).

> Toujours un titre et des axes legendes : un graphique nu est inexploitable pour un decideur.

Apres execution, lis les graphiques ci-dessous : l'histogramme confirme une distribution etiree a droite (beaucoup de petits paniers, quelques gros), les barres classent les villes par CA, et le boxplot revele des points isoles au-dessus des moustaches = **paniers aberrants** a surveiller (la boite va de Q1 a Q3, le trait central est la mediane).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Graphique 1 : histogramme de la distribution des montants
axes[0].hist(ventes["montant"], bins=50, color="#4C72B0", edgecolor="white")
axes[0].set_title("Distribution des montants")
axes[0].set_xlabel("Montant (euros)")
axes[0].set_ylabel("Nombre de ventes")

# Graphique 2 : diagramme en barres du CA par ville
ca_ville = ventes.groupby("ville")["montant"].sum().sort_values()
axes[1].barh(ca_ville.index, ca_ville.values, color="#55A868")
axes[1].set_title("Chiffre d'affaires par ville")
axes[1].set_xlabel("CA (euros)")

# Graphique 3 : boxplot des montants par categorie (reperer les outliers)
categories = ventes["categorie"].unique()
donnees = [ventes.loc[ventes["categorie"] == c, "montant"] for c in categories]
axes[2].boxplot(donnees, labels=categories)
axes[2].set_title("Dispersion par categorie")
axes[2].set_ylabel("Montant (euros)")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

---
## 🎯 A toi de jouer #1 — Filtrer les ventes suspectes

Le controle qualite / anti-fraude : isole les lignes incoherentes avec un masque booleen.

In [ ]:
# TODO : construis un masque booleen pour reperer les ventes suspectes.
# Une vente est suspecte si : marge negative OU remise > 0.8 OU montant negatif.
# Rappel : combine les conditions avec | (OU), chaque condition entre parentheses.
#
# suspectes = ventes[ (...) | (...) | (...) ]
# print("Nombre de ventes suspectes :", len(suspectes))

# Ecris ton code ici :


## 🎯 A toi de jouer #2 — Le panier moyen en e-commerce

La colonne `type` distingue `Magasin` et `E-commerce`. Filtre puis calcule.

In [ ]:
# TODO :
# 1) Filtre les ventes ou type == "E-commerce"
# 2) Calcule et affiche le panier MOYEN (mean) et le panier MEDIAN de ce canal
# 3) Compare-le au panier median global (calcule plus haut). Que constates-tu ?

# Ecris ton code ici :


## 🎯 A toi de jouer #3 — CA par mois

Maintenant que `date` est une vraie date, on peut extraire le mois avec `.dt.month` et grouper dessus.

In [ ]:
# TODO :
# 1) Cree une colonne 'mois' a partir de ventes["date"].dt.month
# 2) Calcule le CA total (sum du montant) par mois avec groupby
# 3) Trace un graphique en barres du CA par mois (plt.bar) avec titre et axes

# Ecris ton code ici :


## Synthese — ce que disent les donnees

En quelques lignes de pandas, on est passe du chaos a des constats actionnables :

- **Qualite** : le fichier est propre (0 NaN, 0 doublon), mais la colonne `date` arrivait en **texte** &rarr; convertie en vraie date pour toute analyse temporelle.
- **Panier typique** : la moyenne du `montant` depasse la mediane &rarr; distribution **etiree a droite**. On communique sur la **mediane** aupres du gerant.
- **Geographie du CA** : le `groupby("ville")` classe les magasins ; quelques villes concentrent l'essentiel du chiffre d'affaires.
- **Outliers** : le boxplot revele des paniers anormalement gros &rarr; **controle recommande**.

> 🧭 **L'etape la plus importante est la conclusion.** Un Data Analyst n'est pas paye pour produire des graphiques, mais pour produire des **constats** qui eclairent une decision.

**Pour aller plus loin** : reprends le cours `01-eda-pandas.md` (TP1 a TP6) et le module `02-statistiques-appliquees.md`.